In [9]:
import env
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Callable, Tuple
from pathlib import Path
from models_config import arch, get_stage_configs
from sample_tensors import b64_to_tensor
from ds.cladder import load_cladder_v1_5, CLadderLoaderConfig, CLadderSample
from InProgressInstance import gather_instances, InProgressInstance

do_models = [
    "Qwen/Qwen3-4B",
    "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen/Qwen3-8B",
    "meta-llama/Llama-3.1-8B-Instruct"
]
model_ablations = ["cadgn", "adgn", "gcn", "gat", "dec"]


In [10]:
## Load full dataset for comparison's purposes
dsConfig = CLadderLoaderConfig(rung_filter=None, query_types=None, skip_unparseable=True)

org_ds = load_cladder_v1_5(dsConfig)

## Load models (we need the EncoderHead specifically)
s1c, s2c, evals2c = get_stage_configs(40)
save_path = Path("./saves/ablations/")
save_path.mkdir(parents=True, exist_ok=True)

in_progress = gather_instances(
    s1config=s1c,
    arch_params=arch,
    ablations=model_ablations,
    llm_models=do_models,
    save_path=save_path,
    ip=True
)

Processing 10112 rows...
Loaded 8532 graphs, skipped 1580 unparseable rows.
Reloading from checkpoint for Stage 1 (ablation=cadgn(0), MMD Weight=0.0)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

GPU: 0.00GB allocated / 0.00GB reserved


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

GPU: 0.00GB allocated / 0.00GB reserved


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

GPU: 0.00GB allocated / 0.00GB reserved


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

GPU: 0.00GB allocated / 0.00GB reserved
	>> Complete!
Reloading from checkpoint for Stage 1 (ablation=cadgn(0), MMD Weight=1.0)
Moving Qwen/Qwen3-4B's embed_layer from cpu to cpu
GPU: 0.00GB allocated / 0.00GB reserved
Moving meta-llama/Llama-3.2-3B-Instruct's embed_layer from cpu to cpu
GPU: 0.00GB allocated / 0.00GB reserved
Moving Qwen/Qwen3-8B's embed_layer from cpu to cpu
GPU: 0.00GB allocated / 0.00GB reserved
Moving meta-llama/Llama-3.1-8B-Instruct's embed_layer from cpu to cpu
GPU: 0.00GB allocated / 0.00GB reserved
	>> Complete!
Reloading from checkpoint for Stage 1 (ablation=adgn(1), MMD Weight=0.0)
Moving Qwen/Qwen3-4B's embed_layer from cpu to cpu
GPU: 0.00GB allocated / 0.00GB reserved
Moving meta-llama/Llama-3.2-3B-Instruct's embed_layer from cpu to cpu
GPU: 0.00GB allocated / 0.00GB reserved
Moving Qwen/Qwen3-8B's embed_layer from cpu to cpu
GPU: 0.00GB allocated / 0.00GB reserved
Moving meta-llama/Llama-3.1-8B-Instruct's embed_layer from cpu to cpu
GPU: 0.00GB allocated

In [ ]:
## Create a container object to store per-sample information
from ds.cladder import CLadderSample, CLadderDataset
from typing import Any, Optional
class SampleExpression:
    def __init__(self,
                 sample_id: int, ablation: str, mmd_val: int,
                 pred_embeds: list[float], pred_logits: list[float], metrics: dict):
        self.sample_id: int = sample_id
        self.ablation: str = ablation
        self.mmd_val: int = mmd_val
        self.pred_embeds: list[float] = pred_embeds
        self.pred_logits: list[float] = pred_logits

        for k, v in metrics.items():
            setattr(self, "m__"+k, v)



class SampleCollection:
    def __init__(self, sample: CLadderSample):
        self.sample = sample
        self.expressions = []

    def __add__(self, other: SampleExpression) -> "PerSample":
        if other.sample_id == self.sample.sample_id:
            self.expressions.append(other)
        return self

    def to_pandas(self, columns=None) -> pd.DataFrame:
        data = []
        for expr in self.expressions:
            row = {col: getattr(expr, col) for col in columns}
            data.append(row)

        df = pd.DataFrame(data)
        # explicitly set
        for col in ['pred_embeds', 'pred_logits']:
            if col in df.columns:
                df[col] = df[col].astype(object)
        return df


train, vald = CLadderDataset.from_samples_split(
    samples=org_ds,
    val_size=0.2,
    stratify=True
)

complete_val = vald.as_dataloader(shuffle=False)

samples: list[SampleCollection] = [ SampleCollection(s) for s in complete_val ]



In [4]:
from sklearn.metrics import roc_auc_score, average_precision_score
import torch

def edge_metrics(pred_logits, true_edge_index, num_nodes):
    # Build ground truth binary adjacency vector
    adj = torch.zeros(num_nodes * num_nodes)
    for src, dst in true_edge_index.T:
        adj[src * num_nodes + dst] = 1.0

    probs = torch.sigmoid(pred_logits).cpu().numpy()
    true  = adj.cpu().numpy()

    return {
        "auroc":  roc_auc_score(true, probs),
        "auprc":  average_precision_score(true, probs),  # better for sparse graphs
        "precision_at_k": ...,  # top-k predicted edges that are correct
    }

In [7]:
do_models = [
    "Qwen/Qwen3-4B",
    "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen/Qwen3-8B",
    "meta-llama/Llama-3.1-8B-Instruct"
]
model_ablations = ["cadgn", "adgn", "gcn", "gat", "dec"]
mmd_vals = [0.0, 1.0]

stage1_output_graph_path = Path("./ablate_results/samples/stage1/")
stage1_output_graph_path.mkdir(parents=True, exist_ok=True)

graph_preds = {}
for a_id, abl in enumerate(model_ablations):
    for im, mmd_val in enumerate(mmd_vals):
        mmd_int = int(mmd_val)
        mmd_str = str(mmd_int)
        fn = f"{a_id}_{abl}_all_{mmd_str}_s1-graph-preds.json"
        with open(stage1_output_graph_path / fn,"r") as f:
            content = json.load(f)

        # convert the b64 encoded logits/tensors back to numpy
        for i, sample in enumerate(content):
            # (num_nodes, token_count, llm_dim)
            sample["pred_embeds"] = {k: b64_to_tensor(d) for k, d in sample["pred_embeds"].items()}

            # flat adjacency matrix
            sample["pred_logits"] = {k: b64_to_tensor(d) for k, d in sample["pred_logits"].items()}

            k = (sample["sample_id"], mmd_int, abl)
            graph_preds[k] = sample
            if i == 0:
                embeds_shape = {k: t.shape for k,t in sample["pred_embeds"].items()}
                edge_logits_shape = {k: t.shape for k,t in sample["pred_logits"].items()}

                print(k, sample["graph_id"], "\n\tEmbeds Shape: ", embeds_shape, "\n\tEdge Logits Shape: ",  edge_logits_shape)



(12210, 0, 'cadgn') IV 
	Embeds Shape:  {'Qwen/Qwen3-4B': (4, 6, 2560), 'meta-llama/Llama-3.2-3B-Instruct': (4, 7, 3072), 'Qwen/Qwen3-8B': (4, 6, 4096), 'meta-llama/Llama-3.1-8B-Instruct': (4, 7, 4096)} 
	Edge Logits Shape:  {'Qwen/Qwen3-4B': (16,), 'meta-llama/Llama-3.2-3B-Instruct': (16,), 'Qwen/Qwen3-8B': (16,), 'meta-llama/Llama-3.1-8B-Instruct': (16,)}
(12210, 1, 'cadgn') IV 
	Embeds Shape:  {'Qwen/Qwen3-4B': (4, 6, 2560), 'meta-llama/Llama-3.2-3B-Instruct': (4, 7, 3072), 'Qwen/Qwen3-8B': (4, 6, 4096), 'meta-llama/Llama-3.1-8B-Instruct': (4, 7, 4096)} 
	Edge Logits Shape:  {'Qwen/Qwen3-4B': (16,), 'meta-llama/Llama-3.2-3B-Instruct': (16,), 'Qwen/Qwen3-8B': (16,), 'meta-llama/Llama-3.1-8B-Instruct': (16,)}
(12210, 0, 'adgn') IV 
	Embeds Shape:  {'Qwen/Qwen3-4B': (4, 6, 2560), 'meta-llama/Llama-3.2-3B-Instruct': (4, 7, 3072), 'Qwen/Qwen3-8B': (4, 6, 4096), 'meta-llama/Llama-3.1-8B-Instruct': (4, 7, 4096)} 
	Edge Logits Shape:  {'Qwen/Qwen3-4B': (16,), 'meta-llama/Llama-3.2-3B-Instru